# Phase 1: Relational Proposer A/B Test

This notebook runs the head-to-head comparison:
- **Baseline**: ResidualFusionHead (scalar proposer)
- **Phase 1**: RelationalFusionHead (cross-atom attention + pairwise)

Expected improvement: +8-12% recall@K

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Clone/update repo
import os
REPO_DIR = '/content/enzyme_Software'

if os.path.exists(REPO_DIR):
    !cd {REPO_DIR} && git pull
else:
    !git clone https://github.com/Nikku03/enzyme_Software.git {REPO_DIR}

os.chdir(REPO_DIR)
!pip install -e . -q

In [ ]:
# Add nexus to path
import sys
sys.path.insert(0, '/content/enzyme_Software')
sys.path.insert(0, '/content/enzyme_Software/nexus')

In [ ]:
# Verify Phase 1 modules
from enzyme_software.liquid_nn_v2.model.relational_proposer import (
    RelationalSelfAttention,
    PairwiseAggregator,
    RelationalProposer,
    RelationalFusionHead
)
import torch

# Quick test
head = RelationalFusionHead(input_dim=128, hidden_dim=96)
x = torch.randn(20, 128)
batch = torch.tensor([0]*10 + [1]*10)
head.set_batch(batch)
out, _ = head(x)
print(f'RelationalFusionHead test: {out.shape}')
print('✓ Phase 1 modules loaded successfully!')

## Training Configuration

We'll run two experiments:
1. Baseline (use_relational_proposer=False)
2. Phase 1 (use_relational_proposer=True)

In [ ]:
# Base config for both experiments
BASE_CONFIG = {
    # Use cleaned dataset
    'data_path': '/content/enzyme_Software/data/cleaned/cyp3a4_cleaned_dataset.json',
    
    # Training params
    'epochs': 30,
    'batch_size': 8,
    'learning_rate': 1e-3,
    
    # Model params
    'hidden_dim': 128,
    'som_branch_dim': 128,
    'som_head_hidden_dim': 96,
    
    # Disable complex features for clean comparison
    'use_nexus_bridge': False,
    'use_topk_reranker': False,
}

# Experiment A: Baseline
BASELINE_CONFIG = {
    **BASE_CONFIG,
    'experiment_name': 'phase1_baseline',
    'use_relational_proposer': False,
}

# Experiment B: Relational Proposer
RELATIONAL_CONFIG = {
    **BASE_CONFIG,
    'experiment_name': 'phase1_relational',
    'use_relational_proposer': True,
    'relational_proposer_num_heads': 4,
    'relational_proposer_num_layers': 2,
    'relational_proposer_use_pairwise': True,
}

## Run Experiments

Use your existing training script with the configs above.

Example integration with `colab_train_hybrid_lnn.py`:

In [ ]:
# Option 1: Modify your existing training script to accept use_relational_proposer
# 
# In colab_train_hybrid_lnn.py, add to config:
#   config.use_relational_proposer = args.use_relational_proposer
#
# Then run:
#   !python scripts/colab_train_hybrid_lnn.py --use_relational_proposer

# Option 2: Direct config modification
from enzyme_software.liquid_nn_v2.config import ModelConfig

# Create relational config
config = ModelConfig(
    use_relational_proposer=True,
    relational_proposer_num_heads=4,
    relational_proposer_num_layers=2,
    relational_proposer_use_pairwise=True,
    # ... other params from your usual config
)

print(f'Config created:')
print(f'  use_relational_proposer: {config.use_relational_proposer}')
print(f'  relational_proposer_num_heads: {config.relational_proposer_num_heads}')

## Expected Results

Based on pairwise probe experiments:

| Metric | Baseline | Phase 1 (Expected) |
|--------|----------|--------------------|
| Recall@6 | ~0.54 | ~0.62-0.66 |
| Recall@12 | ~0.70 | ~0.78-0.82 |
| Top-1 | ~0.49 | ~0.55-0.60 |

Key insight: The pairwise probe achieved 77% accuracy on the same embeddings where the scalar head achieves only 15%. The relational proposer captures this signal.

In [ ]:
# After training both experiments, compare results:
import json
from pathlib import Path

def load_results(experiment_name):
    # Adjust path to your checkpoint directory
    results_path = Path(f'/content/drive/MyDrive/enzyme_hybrid_lnn/checkpoints/{experiment_name}')
    
    # Load final metrics
    metrics_file = results_path / 'final_metrics.json'
    if metrics_file.exists():
        with open(metrics_file) as f:
            return json.load(f)
    return None

# Compare (run after training)
# baseline_results = load_results('phase1_baseline')
# relational_results = load_results('phase1_relational')
# 
# print('Comparison:')
# print(f'  Baseline recall@6: {baseline_results["recall_at_6"]:.3f}')
# print(f'  Relational recall@6: {relational_results["recall_at_6"]:.3f}')
# print(f'  Improvement: {relational_results["recall_at_6"] - baseline_results["recall_at_6"]:.3f}')